[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SivanSchwartz/representation-learning-course/blob/master/tutoring/session01/03_lightning.ipynb)

# 03 · PyTorch Lightning: you write the methods, the Trainer calls them

Fell behind on 02? Run `git pull --no-edit` and open `tutoring/session01/solutions/02_bridge_to_pytorch.ipynb`.

In [ ]:
# Setup: Colab install, data and loaders (the same as notebook 02). Just run it.
import io
import logging
import os
import subprocess
import sys
import urllib.request
import warnings

if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytorch-lightning"], check=True)

import numpy as np
import pytorch_lightning as pl
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# Quieter output. num_workers=0 is on purpose: worker processes break notebooks on Windows and macOS.
warnings.filterwarnings("ignore", ".*does not have many workers.*")
warnings.filterwarnings("ignore", ".*LeafSpec.*")
warnings.filterwarnings("ignore", ".*GPU available but not used.*")
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

DATA_URL = "https://raw.githubusercontent.com/SivanSchwartz/representation-learning-course/master/tutoring/data/mnist_small.npz"


def load_mnist():
    """The local copy if there is one (your laptop), otherwise download it (Colab)."""
    for path in ["../data/mnist_small.npz", "../../data/mnist_small.npz"]:
        if os.path.exists(path):
            return dict(np.load(path))
    return dict(np.load(io.BytesIO(urllib.request.urlopen(DATA_URL).read())))


data = load_mnist()
X = torch.tensor(data["X_train"], dtype=torch.float32).reshape(-1, 784) / 255
y = torch.tensor(data["y_train"], dtype=torch.long)
X_test = torch.tensor(data["X_test"], dtype=torch.float32).reshape(-1, 784) / 255
y_test = torch.tensor(data["y_test"], dtype=torch.long)
torch.manual_seed(0)
train_loader = DataLoader(TensorDataset(X[:5000], y[:5000]), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(X[5000:], y[5000:]), batch_size=256)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256)

## 1. Build your own (tiny) Trainer

In notebook 02, the training loop called the model. Let's flip that around. A `MiniTrainer` owns the 🔧 plumbing, and it calls methods with **agreed names** on whatever model you give it:
- `model.configure_optimizers()`: "which optimizer do you want?"
- `model.training_step(batch, batch_idx)`: "here's a batch, give me the loss"

In [ ]:
# ✍️ The plumbing from notebook 02, packed into a class. It never mentions Linear, ReLU or cross_entropy.

In [ ]:
# ✍️ The model holds only the 🔬 science, as methods with the names MiniTrainer expects

Notice what happened: **you never called `training_step`**. You wrote it, and the trainer called it. This is *inversion of control* ("don't call us, we'll call you"), the same idea as `forward` in `TinyModule`.

In [ ]:
# 🤔 Predict: what happens if we give MiniTrainer a model that doesn't follow the agreement?
plain_model = nn.Sequential(nn.Linear(784, 10))
MiniTrainer(max_epochs=1).fit(plain_model, train_loader)

`'Sequential' object has no attribute 'configure_optimizers'`. The trainer relies on the method names. Lightning works the same way, so **a typo in a hook name** (`train_step`, `configure_optimizer`) means Lightning can't find your method. Depending on which hook it is, Lightning either raises an error like this one or quietly never calls your method.

## 2. Swap in the real thing: `LightningModule` + `pl.Trainer`

Change the parent class, add a `validation_step`, and give the model to Lightning's `Trainer`. Your methods stay the same.

In [ ]:
# ✍️ Same methods. Only the parent class changes, and we add validation_step.

In [ ]:
# ✍️ logger=False for now: notebook 04 turns logging on.

Lightning also ran the validation loop for you, switched `eval()`/`train()` and `no_grad()` on and off at the right times, and printed a model summary and a progress bar. That's all the 🔧 plumbing from notebook 02, and the plumbing you didn't write in your exercise. `pl.LightningModule` is itself a child of `nn.Module`, so everything from notebook 02 still works: `lit_model(x)`, `lit_model.parameters()`, `print(lit_model)`.

## 3. Hooks: when does Lightning call what?

Besides the `*_step` methods, a `LightningModule` has dozens of empty hook methods (`on_fit_start`, `on_train_epoch_start`, …). Lightning calls them at fixed moments. Override one, and your code runs at that moment, just like overriding `forward` in notebook 01.

We inherit from `LitNN` and override five hooks to print when they're called:

In [ ]:
# ✍️ A child class that only overrides hooks

In [ ]:
# 🤔 Predict: which hook prints right after on_fit_start? Training or validation?
quick = pl.Trainer(max_epochs=2, limit_train_batches=5, limit_val_batches=2, accelerator="cpu",
                   logger=False, enable_checkpointing=False, enable_progress_bar=False, enable_model_summary=False)
quick.fit(ChattyNN(), train_loader, val_loader)

Two surprises:
1. **Validation runs first.** It's the *sanity check*: before a long training run starts, Lightning runs two validation batches so a bug in `validation_step` crashes now, not after an hour of training. (`num_sanity_val_steps=0` turns it off.)
2. Each epoch's validation runs **before** `on_train_epoch_end`.

`self.current_epoch` is an attribute that the Trainer keeps up to date on *your* model. The course's `callbacks.py` uses the same hooks, only on a `Callback` class instead of the model.

### ✏️ Exercise 3: add a test step (≈6 min)

`TestableNN` inherits everything from `LitNN`. Add a `test_step(self, batch, batch_idx)` that:
1. computes the predictions, `self(x).argmax(dim=1)`;
2. computes the batch accuracy, `(preds == y).float().mean()`;
3. logs it with `self.log("test_acc", acc)`.

The check cell trains your model and calls `trainer.test(...)`, which calls your `test_step`.

In [ ]:
# ✏️ Your code here
class TestableNN(LitNN):
    def test_step(self, batch, batch_idx):
        ...

In [ ]:
# --- Check (do not edit) ---
torch.manual_seed(0)
testable = TestableNN()
check_trainer = pl.Trainer(max_epochs=2, accelerator="cpu", logger=False, enable_checkpointing=False,
                           enable_progress_bar=False, enable_model_summary=False)
check_trainer.fit(testable, train_loader, val_loader)
results = check_trainer.test(testable, test_loader, verbose=False)
assert "test_acc" in results[0], "Finish the exercise above first ☝️ (test_step should call self.log('test_acc', acc))"
assert results[0]["test_acc"] > 0.85, f"Test accuracy is only {results[0]['test_acc']:.3f}. Check how you compute preds and acc."
print(f"✅ Test accuracy: {results[0]['test_acc']:.3f}")

## 4. Where's the training loop now?

Inside `pl.Trainer.fit`. It's the same loop as `MiniTrainer`'s, plus many more hooks, devices, precision, checkpoints, and early stopping. The arguments you pass to `pl.Trainer(...)` switch those features on and off, and **none of them require changing your model class**. For example, lec5's `train.py` builds its `Trainer` with:

```python
trainer = pl.Trainer(logger=logger, accelerator="auto", devices=1, max_epochs=num_epochs,
                     precision="16-mixed", callbacks=[EarlyStopping(monitor="val_loss"), ...])
```

## 5. The same idea for data: `LightningDataModule`

The model's methods describe *how to learn*. A `LightningDataModule`'s methods describe *where the data comes from*. The course's `MnistDataModule` (`lec2/datamodule.py` to `lec5/datamodule.py`) has this shape.

In [ ]:
# ✍️ Data as hooks too: Lightning calls setup() once, then asks for each loader when it needs it

In [ ]:
# ✍️ Now fit() takes the data module instead of loaders, exactly like the course's train.py

🔄 **Checkpoint:** run `git pull --no-edit` to get `tutoring/session01/solutions/03_lightning.ipynb`.